# 1. Tujuan

Notebook ini melakukan training model klasifikasi final untuk kondisi keuangan bulanan personal user menggunakan dataset labeling baru.

## Dataset dan target
- Dataset: FINARY_MONTHLY_CONDITION_DATASET.csv
- Target: monthly_financial_condition
- Label mapping:
  - 0 = survival
  - 1 = stable
  - 2 = growth

Fokus notebook ini hanya training, evaluasi, dan ekspor artifact final (tanpa integrasi FastAPI pada tahap ini).

## 2. Imports

Import library utama untuk preprocessing, TensorFlow custom training loop, evaluasi, dan export artifact.

In [1]:
import json
import random
import warnings
from datetime import datetime
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import tensorflow as tf
import os
import shutil
import zipfile

from sklearn.metrics import classification_report, confusion_matrix, f1_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import RobustScaler
from sklearn.utils.class_weight import compute_class_weight

warnings.filterwarnings("ignore")
print("TensorFlow:", tf.__version__)

TensorFlow: 2.21.0


## 3. Konfigurasi

Konfigurasi training final dan path artifact.

In [2]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

DATA_PATH = Path("FINARY_MONTHLY_CONDITION_DATASET.csv")
ARTIFACT_DIR = Path("artifacts")
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

TARGET = "monthly_financial_condition"
LABEL_MAPPING = {"0": "survival", "1": "stable", "2": "growth"}
EXCLUDE_FEATURE_COLS = {
    "monthly_financial_condition",
    "monthly_financial_condition_label",
    "financial_scenario",
}

EPOCHS = 150
BATCH_SIZE = 128
PATIENCE = 20
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-4
CLIP_NORM = 1.0
MIN_LR = 1e-5
LR_PATIENCE = 6
LR_FACTOR = 0.5

print("DATA_PATH:", DATA_PATH)
print("TARGET:", TARGET)

DATA_PATH: FINARY_MONTHLY_CONDITION_DATASET.csv
TARGET: monthly_financial_condition


## 4. Load Dataset

Load dataset hasil labeling final untuk training klasifikasi.

In [3]:
df = pd.read_csv(DATA_PATH)
print("Rows:", len(df), "Cols:", df.shape[1])
display(df.head(3))

Rows: 3000 Cols: 45


,monthly_income,monthly_expense_total,savings_rate,budget_goal,financial_scenario,credit_score,debt_to_income_ratio,loan_payment,investment_amount,subscription_services,...,category_Investments,category_Rent,category_Transportation,category_Utilities,cash_flow_status_Neutral,cash_flow_status_Positive,financial_stress_level_Low,financial_stress_level_Medium,monthly_financial_condition,monthly_financial_condition_label
0,3119.58,3212.07,0.38,3676.11,0,721.0,0.56,125.77,689.22,3,...,True,False,False,False,False,True,True,False,0,survival
1,3262.44,3732.81,0.10,2607.17,0,670.0,0.42,454.19,360.34,4,...,True,False,False,False,False,True,True,False,0,survival
2,2931.20,3335.58,0.15,3004.14,0,691.0,0.24,971.82,0.00,5,...,False,False,False,False,False,True,True,False,0,survival


## 5. Minimal Dataset Validation

Validasi ringkas: target tersedia, value target valid, dan distribusi kelas awal.

In [4]:
if TARGET not in df.columns:
    raise KeyError(f"Target column '{TARGET}' not found in dataset")

unique_targets = sorted(df[TARGET].dropna().astype(int).unique().tolist())
if unique_targets != [0, 1, 2]:
    raise ValueError(f"Invalid target values. Expected [0,1,2], got {unique_targets}")

if df[TARGET].isna().any():
    raise ValueError("Target contains missing values")

dist = df[TARGET].value_counts().sort_index()
dist_pct = (dist / dist.sum() * 100).round(2)
print("Target distribution (count):")
print(dist)
print("Target distribution (%):")
print(dist_pct)

Target distribution (count):
monthly_financial_condition
0     651
1     438
2    1911
Name: count, dtype: int64
Target distribution (%):
monthly_financial_condition
0    21.7
1    14.6
2    63.7
Name: count, dtype: float64


## 6. Desain Fitur Minimal

In [5]:
df_prep = df.copy()

# Convert boolean columns to integers.
for col in df_prep.columns:
    if df_prep[col].dtype == bool:
        df_prep[col] = df_prep[col].astype(int)

# =============================
# 1) Definisi fitur minimal final
# =============================
RAW_INPUT_FEATURES = [
    "monthly_income",
    "monthly_expense_total",
    "actual_savings",
    "budget_goal",
    "emergency_fund",
]

COMPUTED_FEATURES = [
    "net_cash_flow",
    "expense_ratio",
    "savings_rate",
    "savings_goal_met",
    "spending_efficiency",
]

FINAL_FEATURES = [
    "monthly_income",
    "monthly_expense_total",
    "actual_savings",
    "net_cash_flow",
    "expense_ratio",
    "savings_rate",
    "budget_goal",
    "savings_goal_met",
    "emergency_fund",
    "spending_efficiency",
]

missing_raw = [c for c in RAW_INPUT_FEATURES if c not in df_prep.columns]
if missing_raw:
    print("[INFO] Kolom raw input tidak ada di dataset dan akan diskip:", missing_raw)

# =============================
# 2) Siapkan raw input (isi missing dengan 0)
# =============================
X_raw = df_prep[[c for c in RAW_INPUT_FEATURES if c in df_prep.columns]].copy()
for c in RAW_INPUT_FEATURES:
    if c not in X_raw.columns:
        X_raw[c] = 0.0

X_raw = X_raw.fillna(0.0)

# =============================
# 3) Hitung fitur turunan dengan safe division
# =============================
monthly_income = X_raw["monthly_income"].astype(float)
monthly_expense_total = X_raw["monthly_expense_total"].astype(float)
actual_savings = X_raw["actual_savings"].astype(float)
budget_goal = X_raw["budget_goal"].astype(float)

net_cash_flow = monthly_income - monthly_expense_total

expense_ratio = np.where(monthly_income > 0, monthly_expense_total / monthly_income, 0.0)
savings_rate = np.where(monthly_income > 0, actual_savings / monthly_income, 0.0)

savings_goal_met = (actual_savings >= budget_goal).astype(float)

spending_efficiency = np.where(monthly_expense_total > 0, net_cash_flow / monthly_expense_total, 0.0)

# =============================
# 4) Bangun X_df final
# =============================
X_df = pd.DataFrame(
    {
        "monthly_income": monthly_income,
        "monthly_expense_total": monthly_expense_total,
        "actual_savings": actual_savings,
        "net_cash_flow": net_cash_flow,
        "expense_ratio": expense_ratio,
        "savings_rate": savings_rate,
        "budget_goal": budget_goal,
        "savings_goal_met": savings_goal_met,
        "emergency_fund": X_raw["emergency_fund"].astype(float),
        "spending_efficiency": spending_efficiency,
    }
).astype("float32")

# Target
if TARGET not in df_prep.columns:
    raise KeyError(f"Target column '{TARGET}' not found in dataset")
y = df_prep[TARGET].astype(int)

# Sanity check leakage
if TARGET in X_df.columns:
    raise ValueError("Target leakage detected: target found in features")

feature_columns = X_df.columns.tolist()

print("Fitur final (minimal) untuk training:")
print(feature_columns)
print("Feature count:", len(feature_columns))
display(X_df.head(3))

Fitur final (minimal) untuk training:
['monthly_income', 'monthly_expense_total', 'actual_savings', 'net_cash_flow', 'expense_ratio', 'savings_rate', 'budget_goal', 'savings_goal_met', 'emergency_fund', 'spending_efficiency']
Feature count: 10


,monthly_income,monthly_expense_total,actual_savings,net_cash_flow,expense_ratio,savings_rate,budget_goal,savings_goal_met,emergency_fund,spending_efficiency
0,3119.580078,3212.070068,0.0,-92.489998,1.029648,0.0,3676.110107,0.0,510.579987,-0.028795
1,3262.439941,3732.810059,0.0,-470.369995,1.144177,0.0,2607.169922,0.0,1154.410034,-0.126010
2,2931.199951,3335.580078,0.0,-404.380005,1.137957,0.0,3004.139893,0.0,1433.020020,-0.121232


## 7. Train/Validation/Test Split

Gunakan stratified split 70/15/15 dengan random_state=42.

In [6]:
X_train_df, X_temp_df, y_train, y_temp = train_test_split(
    X_df,
    y,
    test_size=0.30,
    random_state=SEED,
    stratify=y,
)

X_val_df, X_test_df, y_val, y_test = train_test_split(
    X_temp_df,
    y_temp,
    test_size=0.50,
    random_state=SEED,
    stratify=y_temp,
)

def label_dist(series: pd.Series) -> pd.DataFrame:
    vc = series.value_counts().sort_index()
    pct = (vc / vc.sum() * 100).round(2)
    return pd.DataFrame({"count": vc, "pct": pct})

print("Shapes:")
print("Train:", X_train_df.shape, y_train.shape)
print("Val  :", X_val_df.shape, y_val.shape)
print("Test :", X_test_df.shape, y_test.shape)

print("\nTrain distribution:")
display(label_dist(y_train))
print("Val distribution:")
display(label_dist(y_val))
print("Test distribution:")
display(label_dist(y_test))

Shapes:
Train: (2100, 10) (2100,)
Val  : (450, 10) (450,)
Test : (450, 10) (450,)

Train distribution:


,count,pct
monthly_financial_condition,,
0,456,21.71
1,306,14.57
2,1338,63.71


Val distribution:


,count,pct
monthly_financial_condition,,
0,97,21.56
1,66,14.67
2,287,63.78


Test distribution:


,count,pct
monthly_financial_condition,,
0,98,21.78
1,66,14.67
2,286,63.56


## 8. Validasi Singkat Fitur

Sebelum training, lakukan validasi singkat bahwa fitur minimal masih relevan terhadap target.

In [7]:
# =============================
# Validasi singkat kekuatan fitur minimal
# =============================
print("Selected feature list:")
print(feature_columns)
print("Feature count:", len(feature_columns))

print("\nClass distribution (train/val/test):")
print("- train")
display(label_dist(y_train))
print("- val")
display(label_dist(y_val))
print("- test")
display(label_dist(y_test))

# Korelasi numerik terhadap target (indikasi cepat, bukan kausalitas)
# Target diperlakukan sebagai ordinal untuk korelasi sederhana.
corr_series = pd.concat([X_train_df, y_train.rename("target")], axis=1).corr(numeric_only=True)["target"].drop("target").sort_values(ascending=False)
print("\nKorelasi fitur terhadap target (train, Pearson):")
display(corr_series.to_frame("corr"))

# Group mean fitur utama per kelas
main_features_for_check = [
    c
    for c in [
        "net_cash_flow",
        "expense_ratio",
        "actual_savings",
        "spending_efficiency",
        "monthly_income",
        "monthly_expense_total",
    ]
    if c in X_train_df.columns
]

means_by_class = pd.concat([X_train_df[main_features_for_check], y_train.rename("target")], axis=1).groupby("target").mean(numeric_only=True)
print("\nRata-rata fitur utama per kelas (train):")
display(means_by_class)

# Laporan fitur minimal (untuk artifact)
feature_selection_report = {
    "strategy": "minimal_features_only",
    "raw_input_features": RAW_INPUT_FEATURES,
    "computed_features": COMPUTED_FEATURES,
    "final_features": feature_columns,
    "notes": "Tidak menggunakan fitur yang sulit didapat dari user. Fitur turunan dihitung deterministik dengan safe division.",
}

Selected feature list:
['monthly_income', 'monthly_expense_total', 'actual_savings', 'net_cash_flow', 'expense_ratio', 'savings_rate', 'budget_goal', 'savings_goal_met', 'emergency_fund', 'spending_efficiency']
Feature count: 10

Class distribution (train/val/test):
- train


,count,pct
monthly_financial_condition,,
0,456,21.71
1,306,14.57
2,1338,63.71


- val


,count,pct
monthly_financial_condition,,
0,97,21.56
1,66,14.67
2,287,63.78


- test


,count,pct
monthly_financial_condition,,
0,98,21.78
1,66,14.67
2,286,63.56



Korelasi fitur terhadap target (train, Pearson):


,corr
net_cash_flow,0.817482
savings_rate,0.803135
actual_savings,0.734612
monthly_income,0.614364
spending_efficiency,0.466779
savings_goal_met,0.220249
emergency_fund,-0.002282
budget_goal,-0.008397
monthly_expense_total,-0.551237
expense_ratio,-0.784530



Rata-rata fitur utama per kelas (train):


,net_cash_flow,expense_ratio,actual_savings,spending_efficiency,monthly_income,monthly_expense_total
target,,,,,,
0,-771.875610,1.333119,0.000000,-0.204483,2930.218750,3702.094238
1,296.667114,0.920814,296.667114,0.088426,3681.328857,3384.661621
2,1735.566895,0.615707,1735.566895,0.774242,4413.642578,2678.075684


## 8. Scaling

Gunakan RobustScaler, fit hanya pada train set, lalu transform val/test.

In [8]:
scaler = RobustScaler()
X_train_scaled = scaler.fit_transform(X_train_df.values.astype("float32"))
X_val_scaled = scaler.transform(X_val_df.values.astype("float32"))
X_test_scaled = scaler.transform(X_test_df.values.astype("float32"))

input_dim = X_train_scaled.shape[1]
print("input_dim:", input_dim)

input_dim: 10


## 9. Class Imbalance Handling

Hitung class weight otomatis dari train set dan perkuat minoritas secara ringan.

In [9]:
classes = np.array([0, 1, 2], dtype=int)
base_weights = compute_class_weight(class_weight="balanced", classes=classes, y=y_train.values)
class_weight = {int(c): float(w) for c, w in zip(classes, base_weights)}

# Slight boost for minority classes to avoid collapse to growth.
class_weight[0] *= 1.10
class_weight[1] *= 1.15

# Normalize weights around mean=1 for numerical stability.
w_mean = np.mean(list(class_weight.values()))
class_weight = {k: float(v / w_mean) for k, v in class_weight.items()}

print("Class weights:", class_weight)

def to_onehot(y_arr: np.ndarray, num_classes: int = 3) -> np.ndarray:
    y_arr = y_arr.astype(int)
    out = np.zeros((len(y_arr), num_classes), dtype=np.float32)
    out[np.arange(len(y_arr)), y_arr] = 1.0
    return out

y_train_oh = to_onehot(y_train.values, 3)
y_val_oh = to_onehot(y_val.values, 3)
y_test_oh = to_onehot(y_test.values, 3)

Class weights: {0: 1.0461137513414076, 1: 1.6297743648349197, 2: 0.3241118838236729}


## 10. Build Final TensorFlow Model

Bangun Residual MLP (Functional API) untuk klasifikasi 3 kelas.

In [10]:
@tf.keras.utils.register_keras_serializable(package="finary")
class ResidualDenseBlock(tf.keras.layers.Layer):
    def __init__(
        self,
        units: int,
        dropout: float,
        l2: float,
        activation: str = "gelu",
        name: str | None = None,
        **kwargs,
    ):
        super().__init__(name=name, **kwargs)
        reg = tf.keras.regularizers.l2(l2)
        self.units = units
        self.dropout = dropout
        self.l2 = l2
        self.activation = activation

        self.dense1 = tf.keras.layers.Dense(units, kernel_regularizer=reg)
        self.bn1 = tf.keras.layers.BatchNormalization()
        self.drop1 = tf.keras.layers.Dropout(dropout)

        self.dense2 = tf.keras.layers.Dense(units, kernel_regularizer=reg)
        self.bn2 = tf.keras.layers.BatchNormalization()
        self.drop2 = tf.keras.layers.Dropout(dropout)

        self.proj = None

    def get_config(self):
        config = super().get_config()
        config.update(
            {
                "units": self.units,
                "dropout": self.dropout,
                "l2": self.l2,
                "activation": self.activation,
            }
        )
        return config

    def _act(self, x):
        return tf.keras.activations.gelu(x) if self.activation.lower() == "gelu" else tf.nn.relu(x)

    def build(self, input_shape):
        in_units = int(input_shape[-1])
        if in_units != self.units:
            self.proj = tf.keras.layers.Dense(self.units)
        super().build(input_shape)

    def call(self, x, training=False):
        skip = self.proj(x) if self.proj is not None else x

        y = self.dense1(x)
        y = self.bn1(y, training=training)
        y = self._act(y)
        y = self.drop1(y, training=training)

        y = self.dense2(y)
        y = self.bn2(y, training=training)
        y = self._act(y)
        y = self.drop2(y, training=training)

        return skip + y


def build_residual_mlp(input_dim: int) -> tf.keras.Model:
    reg = tf.keras.regularizers.l2(WEIGHT_DECAY)
    inp = tf.keras.Input(shape=(input_dim,), name="features")

    x = tf.keras.layers.Dense(256, kernel_regularizer=reg, name="proj_dense")(inp)
    x = tf.keras.layers.BatchNormalization(name="proj_bn")(x)
    x = tf.keras.layers.Activation(tf.keras.activations.gelu, name="proj_act")(x)
    x = tf.keras.layers.Dropout(0.25, name="proj_dropout")(x)

    x = ResidualDenseBlock(256, dropout=0.25, l2=WEIGHT_DECAY, activation="gelu", name="res_block_1")(x)
    x = ResidualDenseBlock(128, dropout=0.20, l2=WEIGHT_DECAY, activation="gelu", name="res_block_2")(x)
    x = ResidualDenseBlock(64, dropout=0.15, l2=WEIGHT_DECAY, activation="gelu", name="res_block_3")(x)

    x = tf.keras.layers.BatchNormalization(name="head_bn")(x)
    x = tf.keras.layers.Dense(64, kernel_regularizer=reg, name="head_dense")(x)
    x = tf.keras.layers.Activation(tf.keras.activations.gelu, name="head_act")(x)
    x = tf.keras.layers.Dropout(0.15, name="head_dropout")(x)

    out = tf.keras.layers.Dense(3, activation="softmax", name="class_output")(x)
    return tf.keras.Model(inp, out, name="finary_classification_residual_mlp")


model = build_residual_mlp(input_dim)
model.summary()

Model: "finary_classification_residual_mlp"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ features (InputLayer)           │ (None, 10)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ proj_dense (Dense)              │ (None, 256)            │         2,816 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ proj_bn (BatchNormalization)    │ (None, 256)            │         1,024 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ proj_act (Activation)           │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ proj_dropout (Dropout)          │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ res_block_1                     │ (None, 256)            │       133,632 │
│ (ResidualDenseBlock)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ res_block_2                     │ (None, 128)            │        83,328 │
│ (ResidualDenseBlock)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ res_block_3                     │ (None, 64)             │        21,184 │
│ (ResidualDenseBlock)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ head_bn (BatchNormalization)    │ (None, 64)             │           256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ head_dense (Dense)              │ (None, 64)             │         4,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ head_act (Activation)           │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ head_dropout (Dropout)          │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ class_output (Dense)            │ (None, 3)              │           195 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 246,595 (963.26 KB)

 Trainable params: 244,163 (953.76 KB)

 Non-trainable params: 2,432 (9.50 KB)

## TensorBoard & Checkpoint Setup

Setup TensorBoard `logdir`, `tf.summary` writer, dan checkpoint manager untuk training loop kustom.
Letakkan sebelum loop training; ini menulis scalar per-epoch dan menyimpan checkpoint/model kandidat saat metrik membaik.

In [ ]:
# TensorBoard configuration (per-run)
run_id = datetime.now().strftime("%Y%m%d-%H%M%S")
tb_log_dir = ARTIFACT_DIR / "tensorboard" / run_id
tb_log_dir.mkdir(parents=True, exist_ok=True)
writer = tf.summary.create_file_writer(str(tb_log_dir))

# Save run config (hyperparams) for audit
run_info = {
    "run_id": run_id,
    "epochs": EPOCHS,
    "batch_size": BATCH_SIZE,
    "learning_rate": LEARNING_RATE,
    "weight_decay": WEIGHT_DECAY,
}
with open(ARTIFACT_DIR / f"run_config_{run_id}.json", "w", encoding="utf-8") as f:
    json.dump(run_info, f, indent=2)

print("TensorBoard logs will be written to:", tb_log_dir)
print("Start locally: tensorboard --logdir", ARTIFACT_DIR / "tensorboard")

NameError: name 'optimizer' is not defined

## 11. Custom Component

Gunakan custom weighted categorical crossentropy untuk memasukkan class weight ke training loop.

In [ ]:
_ce = tf.keras.losses.CategoricalCrossentropy(from_logits=False, reduction=tf.keras.losses.Reduction.NONE)


def weighted_categorical_crossentropy(class_weight_dict: dict[int, float]):
    w = np.array([class_weight_dict[i] for i in [0, 1, 2]], dtype=np.float32)
    w = w * (3.0 / float(w.sum()))
    w_t = tf.constant(w, dtype=tf.float32)

    def loss(y_true, y_pred):
        ce = _ce(y_true, y_pred)
        sample_w = tf.reduce_sum(y_true * w_t, axis=-1)
        return sample_w * ce

    return loss


loss_fn = weighted_categorical_crossentropy(class_weight)
print("Custom loss ready: weighted categorical crossentropy")

Custom loss ready: weighted categorical crossentropy


## 12. Custom GradientTape Training Loop

Siapkan dataset pipeline, optimizer, train_step, dan val_step tanpa model.fit.

In [ ]:
def make_tf_dataset(X: np.ndarray, y_onehot: np.ndarray, batch_size: int, training: bool) -> tf.data.Dataset:
    ds = tf.data.Dataset.from_tensor_slices((X.astype(np.float32), y_onehot.astype(np.float32)))
    if training:
        ds = ds.shuffle(len(X), seed=SEED, reshuffle_each_iteration=True)
    return ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)


def macro_f1_from_probs(y_true: np.ndarray, probs: np.ndarray) -> float:
    y_pred = np.argmax(probs, axis=1)
    return float(f1_score(y_true, y_pred, average="macro"))


train_ds = make_tf_dataset(X_train_scaled, y_train_oh, BATCH_SIZE, training=True)
val_ds = make_tf_dataset(X_val_scaled, y_val_oh, BATCH_SIZE, training=False)
test_ds = make_tf_dataset(X_test_scaled, y_test_oh, BATCH_SIZE, training=False)

optimizer_cls = getattr(tf.keras.optimizers, "AdamW", None)
if optimizer_cls is not None:
    optimizer = optimizer_cls(learning_rate=LEARNING_RATE, weight_decay=WEIGHT_DECAY, clipnorm=CLIP_NORM)
    optimizer_name = "AdamW"
else:
    optimizer = tf.keras.optimizers.Adam(learning_rate=LEARNING_RATE, clipnorm=CLIP_NORM)
    optimizer_name = "Adam"

train_acc_m = tf.keras.metrics.CategoricalAccuracy(name="train_accuracy")
val_acc_m = tf.keras.metrics.CategoricalAccuracy(name="val_accuracy")


@tf.function
def train_step(xb, yb):
    with tf.GradientTape() as tape:
        probs = model(xb, training=True)
        loss_vals = loss_fn(yb, probs)
        loss = tf.reduce_mean(loss_vals)
        if model.losses:
            loss += tf.add_n(model.losses)

    grads = tape.gradient(loss, model.trainable_variables)
    optimizer.apply_gradients(zip(grads, model.trainable_variables))
    train_acc_m.update_state(yb, probs)
    return loss


@tf.function
def val_step(xb, yb):
    probs = model(xb, training=False)
    loss_vals = loss_fn(yb, probs)
    loss = tf.reduce_mean(loss_vals)
    if model.losses:
        loss += tf.add_n(model.losses)
    val_acc_m.update_state(yb, probs)
    return loss, probs

print("Optimizer:", optimizer_name)

# Create Checkpoint + Manager now that optimizer exists
ckpt_dir = ARTIFACT_DIR / "checkpoints" / run_id
ckpt_dir.mkdir(parents=True, exist_ok=True)
ckpt = tf.train.Checkpoint(optimizer=optimizer, model=model)
ckpt_manager = tf.train.CheckpointManager(ckpt, str(ckpt_dir), max_to_keep=3)
print("Checkpoint manager created at:", ckpt_dir)

Optimizer: AdamW


## 13. Training Per-Epoch

Training manual per-epoch dengan `tf.GradientTape` + early stopping + restore best weights

In [ ]:
best_val_macro_f1 = -1.0
best_val_loss = np.inf
best_epoch = 0
best_weights = None
bad_epochs = 0
lr_bad_epochs = 0

history = []

for epoch in range(1, EPOCHS + 1):
    train_acc_m.reset_state()
    val_acc_m.reset_state()

    # ===================== TRAIN =====================
    train_losses = []
    for xb, yb in train_ds:
        loss = train_step(xb, yb)
        train_losses.append(float(loss.numpy()))

    # ===================== VALIDATION =====================
    val_losses = []
    val_probs_all = []
    val_y_all = []

    for xb, yb in val_ds:
        vloss, probs = val_step(xb, yb)
        val_losses.append(float(vloss.numpy()))
        val_probs_all.append(probs.numpy())
        val_y_all.append(np.argmax(yb.numpy(), axis=1))

    val_probs_all = np.concatenate(val_probs_all, axis=0)
    val_y_all = np.concatenate(val_y_all, axis=0)

    # ===================== METRICS =====================
    train_loss = float(np.mean(train_losses))
    val_loss = float(np.mean(val_losses))
    train_acc = float(train_acc_m.result().numpy())
    val_acc = float(val_acc_m.result().numpy())
    val_macro_f1 = macro_f1_from_probs(val_y_all, val_probs_all)
    lr = float(optimizer.learning_rate.numpy())

    history.append(
        {
            "epoch": epoch,
            "train_loss": train_loss,
            "val_loss": val_loss,
            "train_accuracy": train_acc,
            "val_accuracy": val_acc,
            "val_macro_f1": val_macro_f1,
            "learning_rate": lr,
        }
    )

    # ===================== PRINT =====================
    print(
        f"Epoch {epoch:03d} | "
        f"train_loss={train_loss:.4f} train_acc={train_acc:.4f} | "
        f"val_loss={val_loss:.4f} val_acc={val_acc:.4f} val_macro_f1={val_macro_f1:.4f} | "
        f"lr={lr:.6f}"
    )

    # ===================== TENSORBOARD LOGGING =====================
    try:
        with writer.as_default():
            tf.summary.scalar("train/loss", train_loss, step=epoch)
            tf.summary.scalar("val/loss", val_loss, step=epoch)
            tf.summary.scalar("train/accuracy", train_acc, step=epoch)
            tf.summary.scalar("val/accuracy", val_acc, step=epoch)
            tf.summary.scalar("val/macro_f1", val_macro_f1, step=epoch)
            tf.summary.scalar("learning_rate", lr, step=epoch)
    except Exception as e:
        print("[WARN] TensorBoard write failed:", e)

    # ===================== EARLY STOPPING & LR =====================
    improved_f1 = val_macro_f1 > best_val_macro_f1 + 1e-4
    improved_loss = val_loss < best_val_loss - 1e-4

    if improved_f1:
        best_val_macro_f1 = val_macro_f1
        best_epoch = epoch
        best_weights = model.get_weights()
        bad_epochs = 0
        # Save checkpoint + best model (overwrite previous best)
        try:
            ckpt_manager.save()
            best_model_path = ARTIFACT_DIR / f"classification_model_best_{run_id}.keras"
            model.save(best_model_path)
            globals()["best_model_path"] = str(best_model_path)
        except Exception as e:
            print("[WARN] saving checkpoint/best model failed:", e)
    else:
        bad_epochs += 1

    if improved_loss:
        best_val_loss = val_loss
        lr_bad_epochs = 0
    else:
        lr_bad_epochs += 1

    if lr_bad_epochs >= LR_PATIENCE:
        new_lr = max(float(optimizer.learning_rate.numpy()) * LR_FACTOR, MIN_LR)
        optimizer.learning_rate.assign(new_lr)
        lr_bad_epochs = 0
        print(f"  ReduceLROnPlateau manual -> lr={new_lr:.6f}")

    if bad_epochs >= PATIENCE:
        print(
            f"Early stopping at epoch {epoch} | "
            f"best_epoch={best_epoch} best_val_macro_f1={best_val_macro_f1:.4f}"
        )
        break

# Save last model (final weights) for record
try:
    last_model_path = ARTIFACT_DIR / f"classification_model_last_{run_id}.keras"
    model.save(last_model_path)
    globals()["last_model_path"] = str(last_model_path)
    print("Saved last model to:", last_model_path)
except Exception as e:
    print("[WARN] saving last model failed:", e)

# ===================== RESTORE BEST =====================
if best_weights is not None:
    model.set_weights(best_weights)

# Flush writer and zip tensorboard logs for artifact
try:
    writer.flush()
except Exception:
    pass

tb_zip_path = None
try:
    tb_zip_path = ARTIFACT_DIR / f"tensorboard_{run_id}.zip"
    with zipfile.ZipFile(tb_zip_path, 'w', compression=zipfile.ZIP_DEFLATED) as zf:
        for root, _, files in os.walk(tb_log_dir):
            for fname in files:
                full = os.path.join(root, fname)
                arcname = os.path.relpath(full, start=tb_log_dir.parent)
                zf.write(full, arcname)
    print("Zipped TensorBoard logs to:", tb_zip_path)
except Exception as e:
    print("[WARN] zipping TensorBoard logs failed:", e)

print("Best epoch:", best_epoch)
print("Best val_macro_f1:", round(best_val_macro_f1, 4))

Epoch 001 | train_loss=0.4861 train_acc=0.7833 | val_loss=0.4738 val_acc=0.8956 val_macro_f1=0.8653 | lr=0.001000
Epoch 002 | train_loss=0.3030 train_acc=0.9033 | val_loss=0.4198 val_acc=0.8511 val_macro_f1=0.7593 | lr=0.001000
Epoch 003 | train_loss=0.2788 train_acc=0.9100 | val_loss=0.4351 val_acc=0.8556 val_macro_f1=0.7455 | lr=0.001000
Epoch 004 | train_loss=0.2580 train_acc=0.9257 | val_loss=0.3799 val_acc=0.8978 val_macro_f1=0.8139 | lr=0.001000
Epoch 005 | train_loss=0.2662 train_acc=0.9224 | val_loss=0.5075 val_acc=0.8444 val_macro_f1=0.7182 | lr=0.001000
Epoch 006 | train_loss=0.2894 train_acc=0.9019 | val_loss=0.3935 val_acc=0.8822 val_macro_f1=0.7852 | lr=0.001000
Epoch 007 | train_loss=0.2430 train_acc=0.9352 | val_loss=0.3392 val_acc=0.9133 val_macro_f1=0.8434 | lr=0.001000
Epoch 008 | train_loss=0.2320 train_acc=0.9276 | val_loss=0.4216 val_acc=0.8733 val_macro_f1=0.7764 | lr=0.001000
Epoch 009 | train_loss=0.2174 train_acc=0.9443 | val_loss=0.4127 val_acc=0.8800 val_macr

## 14. Model Evaluation

Evaluasi final mencakup accuracy, macro/weighted F1, precision/recall/F1 per class, confusion matrix, classification report, dan error analysis singkat.

In [ ]:
def predict_probs(ds: tf.data.Dataset) -> tuple[np.ndarray, np.ndarray]:
    probs_all = []
    y_all = []
    for xb, yb in ds:
        probs = model(xb, training=False).numpy()
        probs_all.append(probs)
        y_all.append(np.argmax(yb.numpy(), axis=1))
    return np.concatenate(y_all, axis=0), np.concatenate(probs_all, axis=0)


def acc(y_true: np.ndarray, probs: np.ndarray) -> float:
    return float((np.argmax(probs, axis=1) == y_true).mean())


y_tr, p_tr = predict_probs(train_ds)
y_va, p_va = predict_probs(val_ds)
y_te, p_te = predict_probs(test_ds)

y_pred = np.argmax(p_te, axis=1)
conf = np.max(p_te, axis=1)

train_accuracy = acc(y_tr, p_tr)
val_accuracy = acc(y_va, p_va)
test_accuracy = acc(y_te, p_te)

macro_f1 = float(f1_score(y_te, y_pred, average="macro"))
weighted_f1 = float(f1_score(y_te, y_pred, average="weighted"))

report = classification_report(
    y_te,
    y_pred,
    labels=[0, 1, 2],
    target_names=[LABEL_MAPPING[str(i)] for i in [0, 1, 2]],
    output_dict=True,
    zero_division=0,
)
cm = confusion_matrix(y_te, y_pred, labels=[0, 1, 2]).tolist()

per_class_precision = {str(k): float(report[LABEL_MAPPING[str(k)]]["precision"]) for k in [0, 1, 2]}
per_class_recall = {str(k): float(report[LABEL_MAPPING[str(k)]]["recall"]) for k in [0, 1, 2]}
per_class_f1 = {str(k): float(report[LABEL_MAPPING[str(k)]]["f1-score"]) for k in [0, 1, 2]}

print("Train accuracy:", round(train_accuracy, 4))
print("Validation accuracy:", round(val_accuracy, 4))
print("Test accuracy:", round(test_accuracy, 4))
print("Macro F1:", round(macro_f1, 4))
print("Weighted F1:", round(weighted_f1, 4))
print("Per-class recall:", {LABEL_MAPPING[k]: round(v, 4) for k, v in per_class_recall.items()})
print("Confusion matrix [survival, stable, growth]:")
print(np.array(cm))

# Error analysis
errors = np.where(y_pred != y_te)[0]
print("\nTotal test errors:", len(errors), "out of", len(y_te))

pair_counts = {}
for idx in errors:
    pair = (int(y_te[idx]), int(y_pred[idx]))
    pair_counts[pair] = pair_counts.get(pair, 0) + 1

if pair_counts:
    print("Most common confusion pairs (true->pred):")
    for (t, p), n in sorted(pair_counts.items(), key=lambda kv: kv[1], reverse=True)[:10]:
        print(f"  {LABEL_MAPPING[str(t)]}->{LABEL_MAPPING[str(p)]}: {n}")

if len(errors) > 0:
    show_n = min(10, len(errors))
    sample_err = errors[:show_n]
    err_rows = X_test_df.iloc[sample_err].copy()
    err_rows["true_label"] = [LABEL_MAPPING[str(int(v))] for v in y_te[sample_err]]
    err_rows["pred_label"] = [LABEL_MAPPING[str(int(v))] for v in y_pred[sample_err]]
    err_rows["confidence"] = conf[sample_err]
    display(err_rows[["true_label", "pred_label", "confidence"]].head(show_n))

Train accuracy: 0.9748
Validation accuracy: 0.9511
Test accuracy: 0.9667
Macro F1: 0.9513
Weighted F1: 0.968
Per-class recall: {'survival': 0.949, 'stable': 1.0, 'growth': 0.965}
Confusion matrix [survival, stable, growth]:
[[ 93   5   0]
 [  0  66   0]
 [  0  10 276]]

Total test errors: 15 out of 450
Most common confusion pairs (true->pred):
  growth->stable: 10
  survival->stable: 5


,true_label,pred_label,confidence
426,growth,stable,0.601215
7,growth,stable,0.617420
2658,growth,stable,0.557665
705,growth,stable,0.524647
2583,growth,stable,0.654940
1091,survival,stable,0.525066
2845,growth,stable,0.923027
2959,growth,stable,0.688586
2288,survival,stable,0.694467
1499,growth,stable,0.723446


## 15. Save Final Artifacts

Simpan model, scaler, feature columns, label mapping, dan metrics final ke folder artifacts.

In [ ]:
model_path = ARTIFACT_DIR / "classification_model.keras"
scaler_path = ARTIFACT_DIR / "classification_scaler.joblib"
feat_cols_path = ARTIFACT_DIR / "classification_feature_columns.json"
label_map_path = ARTIFACT_DIR / "classification_label_mapping.json"
metrics_path = ARTIFACT_DIR / "classification_metrics.json"
feature_selection_path = ARTIFACT_DIR / "classification_feature_selection_report.json"

# model and scaler will be saved after performance check (see conditional block below)

with open(feat_cols_path, "w", encoding="utf-8") as f:
    json.dump(feature_columns, f, indent=2)

with open(label_map_path, "w", encoding="utf-8") as f:
    json.dump(LABEL_MAPPING, f, indent=2)

with open(feature_selection_path, "w", encoding="utf-8") as f:
    json.dump(feature_selection_report, f, indent=2)

# Export artifact hanya jika performa minimal masih bagus.
# Ambang ini bisa Anda sesuaikan; dipilih konservatif agar tidak mengekspor model yang melemah.
EXPORT_MIN_MACRO_F1 = 0.85

# Optional: include tensorboard zip in metrics if present
tb_zip_path = globals().get("tb_zip_path", None)
run_id_val = globals().get("run_id", None)
best_model_path = globals().get("best_model_path", None)
last_model_path = globals().get("last_model_path", None)

metrics = {
    "model_name": model.name,
    "dataset_name": DATA_PATH.name,
    "target_column": TARGET,
    "label_mapping": LABEL_MAPPING,
    "feature_count": int(len(feature_columns)),
    "feature_columns": feature_columns,
    "feature_selection": feature_selection_report,
    "class_distribution": {
        "train": label_dist(y_train).to_dict(orient="index"),
        "val": label_dist(y_val).to_dict(orient="index"),
        "test": label_dist(y_test).to_dict(orient="index"),
    },
    "train_accuracy": float(train_accuracy),
    "val_accuracy": float(val_accuracy),
    "test_accuracy": float(test_accuracy),
    "macro_f1": float(macro_f1),
    "weighted_f1": float(weighted_f1),
    "per_class_precision": per_class_precision,
    "per_class_recall": per_class_recall,
    "per_class_f1": per_class_f1,
    "confusion_matrix": cm,
    "loss_function": "weighted_categorical_crossentropy(class_weight)",
    "optimizer": optimizer_name,
    "epochs_trained": int(len(history)),
    "best_epoch": int(best_epoch),
    "notes": "Retrain dengan fitur minimal realistis + fitur turunan deterministik + custom GradientTape.",
    "tensorboard_run_id": run_id_val,
    "tensorboard_zip": str(tb_zip_path) if tb_zip_path is not None else None,
    "best_model_path": best_model_path,
    "last_model_path": last_model_path,
}

with open(metrics_path, "w", encoding="utf-8") as f:
    json.dump(metrics, f, indent=2)

if float(macro_f1) >= EXPORT_MIN_MACRO_F1:
    model.save(model_path)
    joblib.dump(scaler, scaler_path)

    with open(feat_cols_path, "w", encoding="utf-8") as f:
        json.dump(feature_columns, f, indent=2)

    with open(label_map_path, "w", encoding="utf-8") as f:
        json.dump(LABEL_MAPPING, f, indent=2)

    with open(feature_selection_path, "w", encoding="utf-8") as f:
        json.dump(feature_selection_report, f, indent=2)

    print("Saved artifacts:")
    print("-", model_path)
    print("-", scaler_path)
    print("-", feat_cols_path)
    print("-", label_map_path)
    print("-", feature_selection_path)
    print("-", metrics_path)
else:
    print("[WARN] Macro F1 di bawah ambang export.")
    print("- macro_f1:", float(macro_f1))
    print("- ambang  :", EXPORT_MIN_MACRO_F1)
    print("Artifact model/scaler/feature_columns tidak diekspor ulang.")
    print("Namun metrics tetap disimpan ke:", metrics_path)

Saved artifacts:
- artifacts\classification_model.keras
- artifacts\classification_scaler.joblib
- artifacts\classification_feature_columns.json
- artifacts\classification_label_mapping.json
- artifacts\classification_feature_selection_report.json
- artifacts\classification_metrics.json


## 16. Uji Inferensi Lokal (Input IDR)

Pada bagian ini kita meniru pola inference seperti di `finary_insight_model.ipynb`: input user dalam **IDR**, lalu dikonversi ke skala training dengan `UNIT_SCALE` sebelum masuk `scaler` dan model. Ini memastikan jalur inference konsisten dengan produksi.

In [ ]:
import inspect

# Kontrak unit: input user dalam IDR, lalu diskalakan ke unit training.
# (Dataset training berada pada skala training; untuk produksi gunakan IDR -> /UNIT_SCALE)
UNIT_SCALE = 2500.0


def _safe_div(n: float, d: float) -> float:
    return float(n / d) if d is not None and float(d) > 0 else 0.0


def _ensure_residual_dense_block_keras3_compatible():
    cls = globals().get("ResidualDenseBlock", None)
    needs_redefine = cls is None

    if cls is not None:
        sig = inspect.signature(cls.__init__)
        has_kwargs = any(p.kind == inspect.Parameter.VAR_KEYWORD for p in sig.parameters.values())
        needs_redefine = not has_kwargs

    if needs_redefine:
        @tf.keras.utils.register_keras_serializable(package="finary")
        class ResidualDenseBlock(tf.keras.layers.Layer):
            def __init__(
                self,
                units: int,
                dropout: float,
                l2: float,
                activation: str = "gelu",
                name: str | None = None,
                **kwargs,
            ):
                super().__init__(name=name, **kwargs)
                reg = tf.keras.regularizers.l2(l2)
                self.units = units
                self.dropout = dropout
                self.l2 = l2
                self.activation = activation

                self.dense1 = tf.keras.layers.Dense(units, kernel_regularizer=reg)
                self.bn1 = tf.keras.layers.BatchNormalization()
                self.drop1 = tf.keras.layers.Dropout(dropout)

                self.dense2 = tf.keras.layers.Dense(units, kernel_regularizer=reg)
                self.bn2 = tf.keras.layers.BatchNormalization()
                self.drop2 = tf.keras.layers.Dropout(dropout)

                self.proj = None

            def get_config(self):
                config = super().get_config()
                config.update(
                    {
                        "units": self.units,
                        "dropout": self.dropout,
                        "l2": self.l2,
                        "activation": self.activation,
                    }
                )
                return config

            def _act(self, x):
                return tf.keras.activations.gelu(x) if self.activation.lower() == "gelu" else tf.nn.relu(x)

            def build(self, input_shape):
                in_units = int(input_shape[-1])
                if in_units != self.units:
                    self.proj = tf.keras.layers.Dense(self.units)
                super().build(input_shape)

            def call(self, x, training=False):
                skip = self.proj(x) if self.proj is not None else x

                y = self.dense1(x)
                y = self.bn1(y, training=training)
                y = self._act(y)
                y = self.drop1(y, training=training)

                y = self.dense2(y)
                y = self.bn2(y, training=training)
                y = self._act(y)
                y = self.drop2(y, training=training)

                return skip + y

        globals()["ResidualDenseBlock"] = ResidualDenseBlock


def build_features_from_user_idr(payload: dict, *, unit_scale: float = UNIT_SCALE) -> dict:
    """Bangun fitur inference dari input utama user (IDR).

    Kontrak produksi:
    - wajib (5 field): monthly_income, monthly_expense_total, actual_savings, budget_goal, emergency_fund

    Fitur turunan dihitung deterministik:
    - net_cash_flow
    - expense_ratio
    - savings_rate
    - savings_goal_met
    - spending_efficiency

    Output mengikuti urutan `loaded_feature_columns` (harus sama dengan fitur training minimal).
    """

    required = [
        "monthly_income",
        "monthly_expense_total",
        "actual_savings",
        "budget_goal",
        "emergency_fund",
    ]
    missing = [k for k in required if payload.get(k, None) is None]
    if missing:
        raise ValueError(f"Field wajib (5) belum ada: {missing}")

    inc = float(payload.get("monthly_income", 0.0)) / unit_scale
    exp = float(payload.get("monthly_expense_total", 0.0)) / unit_scale

    sav = float(payload.get("actual_savings", 0.0)) / unit_scale
    goal = float(payload.get("budget_goal", 0.0)) / unit_scale
    emg = float(payload.get("emergency_fund", 0.0)) / unit_scale

    net_cf = inc - exp
    expense_ratio = _safe_div(exp, inc)
    savings_rate = _safe_div(sav, inc)
    savings_goal_met = 1.0 if sav >= goal else 0.0
    spending_efficiency = _safe_div(net_cf, exp)

    feats = {c: 0.0 for c in loaded_feature_columns}
    update = {
        "monthly_income": inc,
        "monthly_expense_total": exp,
        "actual_savings": sav,
        "budget_goal": goal,
        "emergency_fund": emg,
        "net_cash_flow": net_cf,
        "expense_ratio": expense_ratio,
        "savings_rate": savings_rate,
        "savings_goal_met": savings_goal_met,
        "spending_efficiency": spending_efficiency,
    }
    for k, v in update.items():
        if k in feats:
            feats[k] = float(v)

    return feats


_ensure_residual_dense_block_keras3_compatible()

loaded_model = tf.keras.models.load_model(
    ARTIFACT_DIR / "classification_model.keras",
    custom_objects={
        "ResidualDenseBlock": ResidualDenseBlock,
        "finary>ResidualDenseBlock": ResidualDenseBlock,
    },
    compile=False,
)
loaded_scaler = joblib.load(ARTIFACT_DIR / "classification_scaler.joblib")
with open(ARTIFACT_DIR / "classification_feature_columns.json", "r", encoding="utf-8") as f:
    loaded_feature_columns = json.load(f)
with open(ARTIFACT_DIR / "classification_label_mapping.json", "r", encoding="utf-8") as f:
    loaded_label_mapping = json.load(f)

# Contoh payload inference (IDR) sesuai kontrak produksi (hanya input utama)
sample_payload_idr = {
    "monthly_income": 10_000_000,
    "monthly_expense_total": 6_000_000,
    "actual_savings": 1_000_000,
    "budget_goal": 5_000_000,
    "emergency_fund": 1_000_000,
}

features = build_features_from_user_idr(sample_payload_idr)
df_input = pd.DataFrame([features]).reindex(columns=loaded_feature_columns, fill_value=0.0).astype("float32")
X_scaled = loaded_scaler.transform(df_input.values)

probs = loaded_model.predict(X_scaled, verbose=0)[0]
pred_id = int(np.argmax(probs))

inference_output = {
    "prediction": loaded_label_mapping[str(pred_id)],
    "confidence": float(np.max(probs)),
    "probabilities": {loaded_label_mapping[str(i)]: float(probs[i]) for i in range(3)},
}
inference_output

{'prediction': 'stable',
 'confidence': 0.9564299583435059,
 'probabilities': {'survival': 6.856779236841248e-06,
  'stable': 0.9564299583435059,
  'growth': 0.04356318712234497}}

In [ ]:
# Uji inferensi berbasis 3 skenario (input utama user, IDR)

scenario_payloads_idr = {
    "survival": {
        "monthly_income": 7_000_000,
        "monthly_expense_total": 8_500_000,
        "actual_savings": 100_000,
        "budget_goal": 2_000_000,
        "emergency_fund": 300_000,
    },
    "stable": {
        "monthly_income": 12_000_000,
        "monthly_expense_total": 9_500_000,
        "actual_savings": 1_200_000,
        "budget_goal": 3_500_000,
        "emergency_fund": 8_000_000,
    },
    "growth": {
        "monthly_income": 25_000_000,
        "monthly_expense_total": 12_500_000,
        "actual_savings": 6_000_000,
        "budget_goal": 8_000_000,
        "emergency_fund": 60_000_000,
    },
}

rows = []
for name, payload in scenario_payloads_idr.items():
    feats = build_features_from_user_idr(payload)
    df_input = pd.DataFrame([feats]).reindex(columns=loaded_feature_columns, fill_value=0.0).astype("float32")
    X_scaled = loaded_scaler.transform(df_input.values)
    p = loaded_model.predict(X_scaled, verbose=0)[0]
    pred_id = int(np.argmax(p))

    rows.append(
        {
            "skenario": name,
            "prediksi": loaded_label_mapping[str(pred_id)],
            "confidence": float(np.max(p)),
            "p_survival": float(p[0]),
            "p_stable": float(p[1]),
            "p_growth": float(p[2]),
        }
    )

hasil_df = pd.DataFrame(rows).sort_values(by="confidence", ascending=False)
print("Hasil uji skenario (input utama user, IDR):")
display(hasil_df)

rows

Hasil uji skenario (input utama user, IDR):


,skenario,prediksi,confidence,p_survival,p_stable,p_growth
2,growth,growth,0.999982,0.000015,0.000002,0.999982
0,survival,survival,0.998133,0.998133,0.001794,0.000074
1,stable,stable,0.988644,0.000777,0.988644,0.010578


[{'skenario': 'survival',
  'prediksi': 'survival',
  'confidence': 0.9981326460838318,
  'p_survival': 0.9981326460838318,
  'p_stable': 0.0017937066731974483,
  'p_growth': 7.360496965702623e-05},
 {'skenario': 'stable',
  'prediksi': 'stable',
  'confidence': 0.9886444807052612,
  'p_survival': 0.000777442823164165,
  'p_stable': 0.9886444807052612,
  'p_growth': 0.010578004643321037},
 {'skenario': 'growth',
  'prediksi': 'growth',
  'confidence': 0.999982476234436,
  'p_survival': 1.545750637887977e-05,
  'p_stable': 2.0304125882830704e-06,
  'p_growth': 0.999982476234436}]

In [ ]:
# Uji inference dengan input MINIMAL sesuai kontrak produksi
# Wajib (5 field): monthly_income, monthly_expense_total, actual_savings, budget_goal, emergency_fund

minimal_payloads_idr = {
    "survival": {
        "monthly_income": 7_000_000,
        "monthly_expense_total": 8_500_000,
        "actual_savings": 100_000,
        "budget_goal": 2_000_000,
        "emergency_fund": 300_000,
    },
    "stable": {
        "monthly_income": 12_000_000,
        "monthly_expense_total": 9_500_000,
        "actual_savings": 1_200_000,
        "budget_goal": 3_500_000,
        "emergency_fund": 8_000_000,
    },
    "growth": {
        "monthly_income": 25_000_000,
        "monthly_expense_total": 12_500_000,
        "actual_savings": 6_000_000,
        "budget_goal": 8_000_000,
        "emergency_fund": 60_000_000,
    },
}

rows = []
for name, payload in minimal_payloads_idr.items():
    feats = build_features_from_user_idr(payload)
    df_input = pd.DataFrame([feats]).reindex(columns=loaded_feature_columns, fill_value=0.0).astype("float32")
    X_scaled = loaded_scaler.transform(df_input.values)
    p = loaded_model.predict(X_scaled, verbose=0)[0]
    pred_id = int(np.argmax(p))

    rows.append(
        {
            "skenario_input_minimal": name,
            "prediksi": loaded_label_mapping[str(pred_id)],
            "confidence": float(np.max(p)),
            "p_survival": float(p[0]),
            "p_stable": float(p[1]),
            "p_growth": float(p[2]),
        }
    )

hasil_minimal_df = pd.DataFrame(rows).sort_values(by="confidence", ascending=False)
print("Hasil uji skenario (input minimal 5 field wajib, IDR):")
display(hasil_minimal_df)

rows

Hasil uji skenario (input minimal 5 field wajib, IDR):


,skenario_input_minimal,prediksi,confidence,p_survival,p_stable,p_growth
2,growth,growth,0.999982,0.000015,0.000002,0.999982
0,survival,survival,0.998133,0.998133,0.001794,0.000074
1,stable,stable,0.988644,0.000777,0.988644,0.010578


[{'skenario_input_minimal': 'survival',
  'prediksi': 'survival',
  'confidence': 0.9981326460838318,
  'p_survival': 0.9981326460838318,
  'p_stable': 0.0017937066731974483,
  'p_growth': 7.360496965702623e-05},
 {'skenario_input_minimal': 'stable',
  'prediksi': 'stable',
  'confidence': 0.9886444807052612,
  'p_survival': 0.000777442823164165,
  'p_stable': 0.9886444807052612,
  'p_growth': 0.010578004643321037},
 {'skenario_input_minimal': 'growth',
  'prediksi': 'growth',
  'confidence': 0.999982476234436,
  'p_survival': 1.545750637887977e-05,
  'p_stable': 2.0304125882830704e-06,
  'p_growth': 0.999982476234436}]

## 17. Ringkasan Akhir

Model ini memprediksi kondisi keuangan bulanan personal dengan target baru `survival/stable/growth`. Target ini selaras dengan arus kas (cashflow) dan sangat berkaitan dengan indikator utama seperti `net_cash_flow`, `expense_ratio`, dan `actual_savings`. Arsitektur yang digunakan adalah TensorFlow Residual MLP (Functional API) dengan komponen kustom `ResidualDenseBlock` serta training loop berbasis `tf.GradientTape`, lalu diekspor menjadi artifact yang siap dipakai untuk inference di produksi.